<a href="https://colab.research.google.com/github/nityaprakash-b/Intelligent-Banking-Assistant/blob/main/Indian_Banking_Intelligence_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
The goal of this project is to build 'BankGPT', an advanced RAG-based AI system specialized in the Indian banking sector for the 2025-2026 fiscal year. This system will involve building a robust pipeline including structured document generation, hierarchical semantic chunking, hybrid retrieval (combining dense FAISS embeddings and sparse BM25 search), and a Llama 3.2 1B/3B inference engine optimized with 4-bit quantization. The final solution will feature a multi-hop query agent for complex financial reasoning, an automated evaluation suite to ensure factual accuracy, and a Gradio-based interactive dashboard for real-time analysis.

## Phase 1: Environment Setup & Package Management

### Subtask:
Install dependencies and set up the modular directory structure and configuration for BankGPT.


**Reasoning**:
First, I need to install the essential Python packages to support high-performance retrieval and quantized model inference in the Colab environment.



In [ ]:
!pip install -q transformers bitsandbytes accelerate faiss-cpu rank_bm25 sentence-transformers gradio pydantic cross-encoder
print('Libraries installed successfully.')

ERROR: Could not find a version that satisfies the requirement cross-encoder (from versions: none)
ERROR: No matching distribution found for cross-encoder
Libraries installed successfully.


**Reasoning**:
I will fix the pip install command by removing the non-existent 'cross-encoder' package (as it is included in sentence-transformers) and then proceed to create the project directory structure.



In [ ]:
import os

# 1. Re-run installation without the invalid 'cross-encoder' package
!pip install -q transformers bitsandbytes accelerate faiss-cpu rank_bm25 sentence-transformers gradio pydantic

# 2. Define and create the modular directory structure
directories = [
    'data/raw',
    'data/processed',
    'models/embeddings',
    'src/retrieval',
    'src/generation',
    'src/utils',
    'logs'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f'Created directory: {directory}')

print('\nEnvironment setup and directory structure completed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.1 MB/s eta 0:00:00
Created directory: data/raw
Created directory: data/processed
Created directory: models/embeddings
Created directory: src/retrieval
Created directory: src/generation
Created directory: src/utils
Created directory: logs

Environment setup and directory structure completed.


## Phase 2: Structured Document Generation

### Subtask:
Generate the knowledge base of synthetic financial documents for the Indian banking sector (2025-2026) with YAML metadata.


**Reasoning**:
I will create a script to generate and save the synthetic documents to the 'data/raw' directory with appropriate metadata.



In [ ]:
import os

docs = [
    {
        'filename': 'glossary.md',
        'content': """---
title: Indian Banking Glossary 2025
category: Reference
date: 2025-01-10
region: India
---
# Indian Banking Glossary

- **Unified Ledger Interface (ULI)**: A cross-institutional platform for seamless credit assessment and data sharing launched by the RBI.
- **Digital Rupee (e-Rupee)**: The Central Bank Digital Currency (CBDC) used for wholesale and retail transactions.
- **LTV Ratio**: Loan-to-Value ratio, crucial for home loan risk assessment.
"""
    },
    {
        'filename': 'vikas_bank_report.md',
        'content': """---
title: Vikas Bank Annual Report FY25-26
category: Financial Report
date: 2025-06-15
region: India
---
# Vikas Bank Annual Performance Summary

Vikas Bank has shown a 15% growth in Net Interest Margin (NIM) for the Q1 of FY25-26. The bank's focus on green hydrogen project financing has expanded its corporate portfolio by ₹5000 crores. Total NPA (Non-Performing Assets) has dropped to 2.1% due to strict AI-driven credit scoring models implemented in early 2025.
"""
    },
    {
        'filename': 'banking_news_2026.md',
        'content': """---
title: RBI Policy Update March 2026
category: News
date: 2026-03-01
region: India
---
# RBI Holds Repo Rate at 6.25%

In the final MPC meeting of FY 2025-26, the Reserve Bank of India decided to keep the repo rate unchanged at 6.25%. Governor Shaktikanta Das highlighted that inflation is stabilizing within the 4% target band, while the ULI (Unified Ledger Interface) adoption has reached 80% among scheduled commercial banks.
"""
    }
]

for doc in docs:
    path = os.path.join('data/raw', doc['filename'])
    with open(path, 'w') as f:
        f.write(doc['content'])
    print(f'Saved: {path}')

print('\nKnowledge base documents generated successfully.')

Saved: data/raw/glossary.md
Saved: data/raw/vikas_bank_report.md
Saved: data/raw/banking_news_2026.md

Knowledge base documents generated successfully.


In [ ]:
expanded_docs = [
    {
        'filename': 'apex_national_bank_fy25.md',
        'content': "---\ntitle: Apex National Bank Annual Report 2025\ncategory: Financial Report\ndate: 2025-07-20\nregion: India\n---\n# Apex National Bank: Performance Highlights\n\nApex National Bank has reported a stellar performance for FY 2024-25. Key highlights include:\n- **Net Profit**: \u20b912,500 crores, a 22% YoY increase.\n- **Retail Loan Growth**: Driven by a 30% surge in digital home loan applications via the 'ApexSmart' app.\n- **Micro-Lending**: The bank has disbursed \u20b92,000 crores to SHGs (Self Help Groups) in rural Maharashtra and Karnataka.\n- **Dividend**: The Board has recommended a dividend of \u20b95 per equity share.\n\n## Future Outlook\nThe bank plans to open 500 new 'Phygital' branches by 2026, combining physical presence with digital kiosks. The focus remains on maintaining a healthy CASA ratio of 42%."
    },
    {
        'filename': 'global_india_bank_2026.md',
        'content': "---\ntitle: Global India Bank Strategic Roadmap 2026\ncategory: Strategy\ndate: 2026-01-05\nregion: India\n---\n# Global India Bank: Vision 2026\n\nGlobal India Bank (GIB) aims to become the leader in cross-border trade finance using the ULI framework. \n\n### Key Strategic Pillars:\n1. **Sustainability**: Committing to Net Zero by 2040. GIB will stop financing new coal-based power plants by 2027.\n2. **Wealth Management**: Launching 'GIB-Elite' for High Net Worth Individuals (HNIs) with AI-driven portfolio rebalancing.\n3. **Cybersecurity**: A dedicated budget of \u20b9800 crores for Quantum-Resistant Encryption (QRE) implementation across all core banking servers.\n\n### Financial Metrics:\n- **Current ROA**: 1.8%\n- **Target ROE**: 16.5% by Q4 2026."
    },
    {
        'filename': 'rbi_guidelines_2025_2026.md',
        'content': "---\ntitle: RBI Master Circular - Consumer Interest Rates\ncategory: Regulatory\ndate: 2025-11-12\nregion: India\n---\n# RBI Guidelines on Interest Rates and Consumer Protection\n\n### 1. External Benchmark Lending Rate (EBLR)\nAll floating rate personal or retail loans (housing, auto, etc.) and floating rate loans to MSMEs extended by banks must be benchmarked to an external rate, such as the RBI Repo Rate or Treasury Bill rates.\n\n### 2. Reset of Interest Rates\nBanks must provide borrowers the option to switch to a fixed interest rate at the time of interest rate reset. Any charges for this switch must be disclosed transparently in the Key Fact Statement (KFS).\n\n### 3. Penal Charges\nPenalty for non-compliance of loan contract terms shall be treated as 'penal charges' and not as 'penal interest'. There shall be no capitalization of penal charges."
    },
    {
        'filename': 'customer_faq_2026.md',
        'content': "---\ntitle: Consumer Banking FAQ 2026\ncategory: FAQ\ndate: 2026-02-10\nregion: India\n---\n# Frequently Asked Questions for Indian Bank Customers\n\n**Q: How does the Unified Ledger Interface (ULI) help me?**\nA: ULI allows banks to access your verified financial data instantly (with your consent). This means your personal loan approval can happen in minutes rather than days, as the bank doesn't need physical documents for credit appraisal.\n\n**Q: What is the maximum insurance on my deposits?**\nA: Under the DICGC act, your deposits (savings, fixed, current) are insured up to \u20b95 lakhs per bank.\n\n**Q: Can I change my EMI date?**\nA: Yes, per RBI 2025 guidelines, banks must offer a flexible 'EMI-Date-Reset' option once a year without heavy processing fees, provided the request is made 15 days in advance."
    }
]

for doc in expanded_docs:
    path = os.path.join('data/raw', doc['filename'])
    with open(path, 'w') as f:
        f.write(doc['content'])
    print(f'Saved: {path}')

print('Additional synthetic documents generated.')

Saved: data/raw/apex_national_bank_fy25.md
Saved: data/raw/global_india_bank_2026.md
Saved: data/raw/rbi_guidelines_2025_2026.md
Saved: data/raw/customer_faq_2026.md
Additional synthetic documents generated.


In [ ]:
# Re-generate the PDF to include all documents (approx 10 pages of content)
create_pdf_from_kb('data/raw', 'outputs/BankGPT_Expanded_KB.pdf')

PDF successfully generated at: outputs/BankGPT_Expanded_KB.pdf


In [ ]:
# Pass through Phase 3: Ingestion
!python src/ingestion.py

# Pass through Phase 4: Indexing
!python src/indexer.py

In [ ]:
import shutil
import os

# Define paths to clear
paths_to_clear = ['data/processed', 'models/embeddings']

for path in paths_to_clear:
    if os.path.exists(path):
        shutil.rmtree(path)
        os.makedirs(path)
        print(f'Cleared and recreated: {path}')

print('\nSystem state is now clean. Re-running ingestion and indexing...')


Cleared and recreated: data/processed
Cleared and recreated: models/embeddings

System state is now clean. Re-running ingestion and indexing...


In [ ]:
# Re-run Phase 3: Ingestion with new expanded data
!python src/ingestion.py

# Re-run Phase 4: Indexing with new expanded data
!python src/indexer.py

Successfully created 11 chunks in data/processed/chunks.json
Loading embedding model: BAAI/bge-small-en-v1.5...
Loading weights: 100% 199/199 [00:00<00:00, 5027.08it/s]
Generating embeddings for FAISS...
Batches: 100% 1/1 [00:03<00:00,  3.32s/it]
Building BM25 index...
Indices saved successfully.


### Pipeline Refreshed
The previous embeddings and chunks have been deleted. The system has now processed the expanded knowledge base (~10 pages).

**Ready for your final approval to move to Phase 6 (LLM Generation).**

### Data Expansion Complete
The knowledge base has been expanded, the PDF is updated at `outputs/BankGPT_Expanded_KB.pdf`, and the search indices have been rebuilt. **Awaiting approval to proceed to Phase 6 inference.**

In [ ]:
!pip install -q fpdf

  Preparing metadata (setup.py) ... done


In [ ]:
import os
from fpdf import FPDF

def create_pdf_from_kb(input_dir, output_path):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.md'):
            pdf.add_page()
            pdf.set_font('Arial', 'B', 16)
            pdf.cell(200, 10, txt=f'Document: {filename}', ln=True, align='C')
            pdf.ln(10)

            file_path = os.path.join(input_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            pdf.set_font('Arial', '', 12)
            # Multi_cell handles long text and wrapping
            pdf.multi_cell(0, 10, txt=content.encode('latin-1', 'replace').decode('latin-1'))

    pdf.output(output_path)
    print(f'PDF successfully generated at: {output_path}')

# Generate the PDF
os.makedirs('outputs', exist_ok=True)
create_pdf_from_kb('data/raw', 'outputs/BankGPT_KnowledgeBase.pdf')

PDF successfully generated at: outputs/BankGPT_KnowledgeBase.pdf


## Phase 3: Hierarchical Document Chunking & Metadata Enrichment

### Subtask:
Implement the `src/ingestion.py` script to process documents into chunks with injected metadata preambles.

In [ ]:
import os
import yaml
import json

# Updated ingestion logic to correctly stringify date objects before JSON serialization
ingestion_script = """
import os
import yaml
import json
import re
from datetime import date

class BankGPTIngestor:
    def __init__(self, chunk_size=600, overlap=50):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def parse_markdown(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        match = re.match(r'^---\\s+(.*?)\\s+---\\s+(.*)', content, re.DOTALL)
        if match:
            metadata = yaml.safe_load(match.group(1))
            body = match.group(2)
        else:
            metadata = {'filename': os.path.basename(file_path)}
            body = content

        # Ensure all metadata values (especially dates) are JSON serializable
        processed_metadata = {}
        for key, value in metadata.items():
            if isinstance(value, date):
                processed_metadata[key] = value.isoformat()
            else:
                processed_metadata[key] = value

        return processed_metadata, body

    def chunk_text(self, text, metadata):
        chunks = []
        preamble = f"Source: {metadata.get('title', 'Unknown')} | Bank: {metadata.get('bank', 'General')} | Date: {metadata.get('date', '2025')} | "

        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk_body = text[start:end]

            chunks.append({
                "text_content": preamble + chunk_body.strip(),
                "metadata_dict": metadata,
                "source_file": metadata.get('filename', 'unknown')
            })
            start += (self.chunk_size - self.overlap)

        return chunks

    def run(self, raw_dir, output_path):
        all_chunks = []
        if not os.path.exists(raw_dir):
            print(f'Error: {raw_dir} not found.')
            return 0

        for filename in os.listdir(raw_dir):
            if filename.endswith('.md'):
                meta, body = self.parse_markdown(os.path.join(raw_dir, filename))
                meta['filename'] = filename
                all_chunks.extend(self.chunk_text(body, meta))

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(all_chunks, f, indent=4)

        return len(all_chunks)

if __name__ == '__main__':
    ingestor = BankGPTIngestor()
    count = ingestor.run('data/raw', 'data/processed/chunks.json')
    print(f'Successfully created {count} chunks in data/processed/chunks.json')
"""

with open('src/ingestion.py', 'w') as f:
    f.write(ingestion_script)

!python src/ingestion.py

Successfully created 3 chunks in data/processed/chunks.json


## Phase 4: Dense & Sparse Indexing (FAISS & BM25)

### Subtask:
Create the `src/indexer.py` module to build and save the hybrid search indices.

In [ ]:
# Debugging the JSON output and re-running the indexer
import json

try:
    with open('data/processed/chunks.json', 'r') as f:
        data = json.load(f)
    print(f'JSON is valid. Number of chunks: {len(data)}')
    # Re-running the indexing script
    !python src/indexer.py
except Exception as e:
    print(f'Error detected in chunks.json: {e}')
    # Display the first few characters of the file for debugging
    with open('data/processed/chunks.json', 'r') as f:
        print('File snippet:', f.read(500))

JSON is valid. Number of chunks: 3
Loading embedding model: BAAI/bge-small-en-v1.5...
Loading weights: 100% 199/199 [00:00<00:00, 4354.93it/s]
Generating embeddings for FAISS...
Batches: 100% 1/1 [00:00<00:00,  1.50it/s]
Building BM25 index...
Indices saved successfully.


## Phase 5: Hybrid Retrieval & Reranking

### Subtask:
Implement `src/retriever.py` to merge dense/sparse results and apply a Cross-Encoder reranker.

In [ ]:
retriever_script = """
import json
import pickle
import faiss
import numpy as np
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# Suppress HF authentication warnings in logs
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

class HybridRetriever:
    def __init__(self, faiss_path, bm25_path, chunks_path, model_name='BAAI/bge-small-en-v1.5'):
        print("Initializing Hybrid Retriever with RRF...")
        self.embed_model = SentenceTransformer(model_name)
        self.rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

        self.faiss_index = faiss.read_index(faiss_path)
        with open(bm25_path, 'rb') as f:
            self.bm25 = pickle.load(f)
        with open(chunks_path, 'r') as f:
            self.chunks = json.load(f)

    def search(self, query, top_k=5, rrf_k=60):
        # 1. Dense Search
        query_emb = self.embed_model.encode([query])
        faiss.normalize_L2(query_emb)
        dense_scores, dense_indices = self.faiss_index.search(query_emb, top_k * 4)

        # 2. Sparse Search
        tokenized_query = query.lower().split()
        sparse_scores = self.bm25.get_scores(tokenized_query)
        sparse_indices = np.argsort(sparse_scores)[::-1][:top_k * 4]

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}

        # Dense Rank scores
        for rank, idx in enumerate(dense_indices[0]):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)

        # Sparse Rank scores
        for rank, idx in enumerate(sparse_indices):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)

        # Sort by RRF score and pick top candidates for reranking
        sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k * 3]
        candidates = [self.chunks[i]['text_content'] for i in sorted_indices]

        # 4. Cross-Encoder Reranking
        model_inputs = [[query, c] for c in candidates]
        rerank_scores = self.rerank_model.predict(model_inputs)

        # Final Sort
        results = sorted(zip(sorted_indices, rerank_scores), key=lambda x: x[1], reverse=True)

        final_chunks = []
        for idx, score in results[:top_k]:
            chunk = self.chunks[idx].copy()
            chunk['rerank_score'] = float(score)
            final_chunks.append(chunk)

        return final_chunks

if __name__ == '__main__':
    retriever = HybridRetriever(
        'models/embeddings/faiss_index.bin',
        'models/embeddings/bm25_index.pkl',
        'data/processed/chunks.json'
    )
    test_query = "How has the RBI handled the repo rate and what is the target inflation?"
    results = retriever.search(test_query)

    print(f"\\nQuery: {test_query}")
    print("-" * 30)
    for i, res in enumerate(results):
        print(f"Rank {i+1} (Score: {res['rerank_score']:.4f}):")
        print(f"{res['text_content'][:200]}...\\n")
"""

with open('src/retriever.py', 'w') as f:
    f.write(retriever_script)

# Run the updated retriever
!python src/retriever.py

Initializing Hybrid Retriever with RRF...
Loading weights: 100% 199/199 [00:00<00:00, 5440.12it/s]
Loading weights: 100% 105/105 [00:00<00:00, 13347.54it/s]

Query: How has the RBI handled the repo rate and what is the target inflation?
------------------------------
Rank 1 (Score: 4.8753):
Source: RBI Policy Update March 2026 | Bank: General | Date: 2026-03-01 | # RBI Holds Repo Rate at 6.25%

In the final MPC meeting of FY 2025-26, the Reserve Bank of India decided to keep the repo rat...

Rank 2 (Score: 4.8753):
Source: RBI Policy Update March 2026 | Bank: General | Date: 2026-03-01 | # RBI Holds Repo Rate at 6.25%

In the final MPC meeting of FY 2025-26, the Reserve Bank of India decided to keep the repo rat...

Rank 3 (Score: -10.9196):
Source: Indian Banking Glossary 2025 | Bank: General | Date: 2025-01-10 | # Indian Banking Glossary

- **Unified Ledger Interface (ULI)**: A cross-institutional platform for seamless credit assessment...

Rank 4 (Score: -11.3728):
Source: Vikas Ban

## Phase 6: Quantized Inference Engine (Llama 3.2)

### Subtask:
Implement `src/generator.py` to orchestrate the RAG pipeline using a 4-bit quantized Llama 3.2 model.

In [ ]:
generator_script = """
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.retriever import HybridRetriever
import os

class BankGPTGenerator:
    def __init__(self, model_id='unsloth/Llama-3.2-3B-Instruct-bnb-4bit'):
        print(f'Loading Quantized LLM: {model_id}...')

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )

        self.retriever = HybridRetriever(
            'models/embeddings/faiss_index.bin',
            'models/embeddings/bm25_index.pkl',
            'data/processed/chunks.json'
        )

    def format_prompt(self, query, context):
        prompt = f\"\"\"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are BankGPT, a professional AI assistant for the Indian Banking Sector (2025-2026).\nUse the provided Context to answer the User Query. Ensure financial accuracy.\nIf the information is not available in the context, clearly state that.\n\n### CONTEXT:\n{context}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\"\"\"
        return prompt

    def generate_response(self, query, max_new_tokens=256):
        search_results = self.retriever.search(query, top_k=3)
        context_text = "\\n\\n".join([r['text_content'] for r in search_results])

        prompt = self.format_prompt(query, context_text)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return full_text.split('assistant')[-1].strip()

if __name__ == '__main__':
    bot = BankGPTGenerator()
    query = "What is the current repo rate and how is ULI adoption progressing?"
    print(f'\\nQuery: {query}')
    print('='*40)
    response = bot.generate_response(query)
    print(f'BankGPT: {response}')
"""

os.makedirs('src', exist_ok=True)
with open('src/generator.py', 'w') as f:
    f.write(generator_script)

print('Generator module src/generator.py created successfully.')

Generator module src/generator.py created successfully.


In [ ]:
# Execute the fixed generator script to load the model and test the RAG pipeline
!python src/generator.py

Traceback (most recent call last):
  File "/content/src/generator.py", line 4, in <module>
    from src.retriever import HybridRetriever
ModuleNotFoundError: No module named 'src'


In [ ]:
# 1. Create __init__.py to ensure 'src' is treated as a package
!touch src/__init__.py

# 2. Re-execute the generator script with the current directory in the PYTHONPATH
import os
os.environ['PYTHONPATH'] = os.getcwd()
!python src/generator.py

Loading Quantized LLM: unsloth/Llama-3.2-3B-Instruct-bnb-4bit...
config.json: 100% 1.47k/1.47k [00:00<00:00, 3.01MB/s]
tokenizer_config.json: 100% 54.7k/54.7k [00:00<00:00, 75.2MB/s]
tokenizer.json: 100% 17.2M/17.2M [00:00<00:00, 22.8MB/s]
special_tokens_map.json: 100% 454/454 [00:00<00:00, 1.81MB/s]
chat_template.jinja: 100% 3.83k/3.83k [00:00<00:00, 7.95MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
model.safetensors: 100% 2.24G/2.24G [00:18<00:00, 124MB/s]
Loading weights: 100% 254/254 [00:00<00:00, 349.61it/s]
generation_config.json: 100% 234/234 [00:00<00:00, 264kB/s]
Initializing Hybrid Retriever with RRF...
Loading weights: 100% 199/199 [00:00<00:00, 4886.23it/s]
Loading weights: 100% 105/105 [00